# 04_12 Recommendation - ALS
Train and evaluate ALS implicit feedback.

[COMMAND_SO]
Command 1

[COMMAND_MUC_DICH]
- Muc tieu nghiep vu: Train ALS va xem ket qua goi y top-N ro rang tren notebook.
- Muc tieu ky thuat: Hien thi metric train/val/test va bang recommendation mau.

In [1]:
import sys, asyncio
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
from pathlib import Path
import json
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd
from IPython.display import display
spark=(SparkSession.builder.appName('04_12_als').master('local[2]').config('spark.sql.shuffle.partitions','16').getOrCreate()) # type: ignore
spark.sparkContext.setLogLevel('WARN')
PROJECT_ROOT=Path.cwd().resolve().parent if Path.cwd().name=='notebooks' else Path.cwd().resolve()
FEATURE_DIR=PROJECT_ROOT/'data'/'processed'/'features'
MODEL_DIR=PROJECT_ROOT/'models'/'recommendation'/'als'
METRIC_DIR=PROJECT_ROOT/'reports'/'model_metrics'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
train_df=spark.read.parquet(str(FEATURE_DIR/'als_train')).select('user_idx','item_idx','rating').dropna()
val_df=spark.read.parquet(str(FEATURE_DIR/'als_val')).select('user_idx','item_idx','rating').dropna()
test_df=spark.read.parquet(str(FEATURE_DIR/'als_test')).select('user_idx','item_idx','rating').dropna()
als=ALS(userCol='user_idx',itemCol='item_idx',ratingCol='rating',implicitPrefs=True,alpha=40.0,rank=64,regParam=0.05,maxIter=15,coldStartStrategy='nan',seed=42)
m=als.fit(train_df)
pred_val=m.transform(val_df)
pred_test=m.transform(test_df)
evaluator=RegressionEvaluator(labelCol='rating',predictionCol='prediction',metricName='rmse')
# Filter NaN predictions (from coldStartStrategy='nan') instead of dropping all rows
from pyspark.sql import functions as F
pred_val_eval=pred_val.where(F.col("prediction").isNotNull() & ~F.isnan(F.col("prediction")) & F.col("rating").isNotNull())
pred_test_eval=pred_test.where(F.col("prediction").isNotNull() & ~F.isnan(F.col("prediction")) & F.col("rating").isNotNull())
val_eval_has_rows=pred_val_eval.limit(1).count()>0
test_eval_has_rows=pred_test_eval.limit(1).count()>0
val_rmse=float(evaluator.evaluate(pred_val_eval)) if val_eval_has_rows else None
test_rmse=float(evaluator.evaluate(pred_test_eval)) if test_eval_has_rows else None
rmse_for_report=test_rmse if test_rmse is not None else val_rmse
metrics={'model_family':'recommendation','model_name':'ALS','val_rmse':val_rmse,'rmse':rmse_for_report,'test_rmse':test_rmse,'train_rows':train_df.count(),'val_rows':val_df.count(),'test_rows':test_df.count(),'val_eval_ready':bool(val_eval_has_rows),'test_eval_ready':bool(test_eval_has_rows),'val_eval_rows':pred_val_eval.count(),'test_eval_rows':pred_test_eval.count()}
print(metrics)
display(pd.DataFrame([metrics]))
m.write().overwrite().save(str(MODEL_DIR))
(METRIC_DIR/'recommendation_als.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
try:
    user_subset=train_df.select('user_idx').dropna().distinct().limit(30)
    rec_df=m.recommendForUserSubset(user_subset,10)
    rec_df.show(5, truncate=False)
    rec_rows=rec_df.limit(30).collect()
    rec_pdf=pd.DataFrame([r.asDict(recursive=True) for r in rec_rows])
    display(rec_pdf)
except Exception as e:
    print(f'Warning: skip recommendation preview due to: {e}')
spark.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 00:04:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/03 00:04:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/03 00:04:41 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/03 00:04:41 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/03 00:04:41 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/04/03 00:04:41 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/04/03 00:04:41 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/04/03 00:04:43 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/03 00:04:43 WARN Ins

{'model_family': 'recommendation', 'model_name': 'ALS', 'val_rmse': 5.3991097175237455, 'rmse': 5.416005715779455, 'test_rmse': 5.416005715779455, 'train_rows': 71277, 'val_rows': 15368, 'test_rows': 15342, 'val_eval_ready': True, 'test_eval_ready': True, 'val_eval_rows': 918, 'test_eval_rows': 916}


,model_family,model_name,val_rmse,rmse,test_rmse,train_rows,val_rows,test_rows,val_eval_ready,test_eval_ready,val_eval_rows,test_eval_rows
0,recommendation,ALS,5.39911,5.416006,5.416006,71277,15368,15342,True,True,918,916


+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_idx|recommendations                                                                                                                                                                       |
+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|40120   |[{29, 0.55246514}, {22, 0.519709}, {7131, 0.51428753}, {10, 0.4474958}, {109, 0.43223414}, {69, 0.4190607}, {131, 0.36730653}, {389, 0.36660114}, {273, 0.36374867}, {17, 0.36284813}]|
|49261   |[{208, 0.7805115}, {26, 0.3853901}, {48, 0.34709418}, {36, 0.2931096}, {92, 0.27505964}, {130, 0.27065116}, {101, 0.25131032}, {393, 0.24550863}, {14, 0.24076504}, {344, 0.23734498}]|
|64601   |[{3430, 0.46147567},

,user_idx,recommendations
0,40120,"[{'item_idx': 29, 'rating': 0.5524651408195496..."
1,49261,"[{'item_idx': 208, 'rating': 0.780511498451232..."
2,64601,"[{'item_idx': 3430, 'rating': 0.46147567033767..."
3,19042,"[{'item_idx': 82, 'rating': 0.8790620565414429..."
4,76662,"[{'item_idx': 1400, 'rating': 0.75609493255615..."
5,69523,"[{'item_idx': 2211, 'rating': 0.63989061117172..."
6,93813,"[{'item_idx': 12, 'rating': 0.3955020904541015..."
7,434,"[{'item_idx': 32, 'rating': 0.7930423021316528..."
8,61854,"[{'item_idx': 36, 'rating': 0.2219003736972808..."
9,91304,"[{'item_idx': 932, 'rating': 0.810022830963134..."
